# Example 3b — Source Term Identification

Reproduces **Figure 4** of Alberts & Bilionis (2023). Simultaneous inference of (D, κ) **and** source f(x), with f represented by a 10-term Karhunen–Loève expansion (squared-exponential kernel C(x, x')=exp(−(x−x')² / 0.18), Nyström on a fine grid). Latent: λ = (log D, log κ, z₁, ..., z₁₀) sampled via nested SGLD.

In [ ]:
# Colab bootstrap — installs CUDA-enabled JAX. No-op locally.
import os, sys
from pathlib import Path
ON_COLAB = 'google.colab' in sys.modules
if ON_COLAB:
    %cd /content
    !nvidia-smi -L || echo 'No GPU detected — Runtime > Change runtime type > GPU'
    !rm -rf /content/pift-od-il-inverse-problems
    !git clone https://github.com/cmhobbs96/pift-od-il-inverse-problems.git /content/pift-od-il-inverse-problems
    assert Path('/content/pift-od-il-inverse-problems/pyproject.toml').exists()
    %cd /content/pift-od-il-inverse-problems
    !pip install -q --upgrade pip
    !pip install -q -e .
    !pip install -q --upgrade "jax[cuda12]"
    os.environ['PIFT_FORCE_BACKEND'] = 'local'
    os.environ['PIFT_FORCE_DEVICE'] = 'gpu'
    os.environ['JAX_PLATFORMS'] = 'cuda'
    os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
    print('Colab bootstrap complete.')
else:
    print('Not on Colab — bootstrap skipped.')

In [ ]:
import jax
print('JAX backend :', jax.default_backend())
print('JAX devices :', jax.devices())
if ON_COLAB:
    assert any(d.platform == 'gpu' for d in jax.devices()), 'CUDA GPU not visible to JAX'
    print('CUDA enabled ✓')

In [ ]:
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == 'examples':
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
DEVICE = 'gpu' if ON_COLAB else 'cpu'
print('Repo root:', ROOT, '| device_preference:', DEVICE)

In [ ]:
from src.pipelines.phase_c_inverse_source import run_phase_c_inverse_source
result = run_phase_c_inverse_source(device_preference=DEVICE)
print('status:', result['status'], '| runtime (s):', round(result.get('runtime_sec', 0), 1))

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg
for path in result.get('artifacts', []):
    if str(path).endswith('.png'):
        fig, ax = plt.subplots(figsize=(11, 8))
        ax.imshow(mpimg.imread(path)); ax.set_title(Path(path).name); ax.axis('off')
        plt.show()

**Expected:** joint D–κ posterior, recovered f(x) with credible band bracketing the true source, and prior/posterior predictive checks at the observation locations.